# 00 - Setup and the shared spine

Lecture section: Setup & conventions  |  Spine term this tutorial changes: none (we fix the stage)

$$\hat{x} = \arg\min_x\ \underbrace{D(Ax,\,y)}_{\text{data fidelity}} + \underbrace{R(x)}_{\text{prior}}$$

This is the single equation the whole lecture restates, one term at a time.
**$D(Ax, y)$** is the *data-fidelity*: it pins our guess to the measurements through
the physics (the forward operator $A$) and the noise model. **$R(x)$** is the *prior*:
what we believe a plausible object looks like, **before** seeing any data.

The lecture arc keeps $D$ **fixed** and makes the prior $R$ progressively smarter:
**Tikhonov** (smoothness) $\rightarrow$ **TV / sparsity** (edges) $\rightarrow$
**PnP / RED** (a learned denoiser). One object, one operator, one metric -- only $R$ changes.

In [1]:
import tutorial_common as tc           # prints the deepinv / torch / device banner
import deepinv as dinv
import torch
import matplotlib.pyplot as plt        # for the hand-drawn palette swatch

tc.set_seed()                          # every notebook is bit-for-bit reproducible
print("spine:", tc.SPINE)              # the LaTeX spine string reused across the deck

deepinv 0.4.1 | torch 2.9.1 | device cpu


spine: $\hat{x} = \arg\min_x\ \underbrace{D(Ax, y)}_{\mathrm{data\ fidelity}} + \underbrace{R(x)}_{\mathrm{prior}}$


## The one hero object $x$

Every tutorial reconstructs the **same** object: a Shepp-Logan phantom in $[0,1]$.
It stands in for the photoacoustic / CT object we want to image. Having one hero
means "did the prior help?" is always answered on the *same* picture.

In [2]:
x = tc.load_hero(128)                                   # (1,1,128,128) phantom in [0,1]
print("x shape:", tuple(x.shape), "| range:", (float(x.min()), float(x.max())))
tc.save_images([x], titles=["x (the object)"], fname="00_hero.png")

x shape: (1, 1, 128, 128) | range: (0.0, 1.0)


saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/00_hero.png


## The deck palette

Throughout the slides we color-code the three actors of the spine so they are
instantly recognizable: the **object $x$** (red), the **operator $A$** (navy),
and the **measurements $y$** (green).

In [3]:
# A tiny swatch figure: one colored rectangle per palette role, with a label.
swatches = [("x", "object  (ground truth)"),
            ("A", "operator  (physics)"),
            ("y", "measurements")]
fig, ax = plt.subplots(figsize=(6, 2.2))
for i, (key, label) in enumerate(swatches):
    ax.add_patch(plt.Rectangle((i, 0), 0.9, 1, color=tc.PALETTE[key]))  # colored box
    ax.text(i + 0.45, -0.18, f"{key}", ha="center", va="top",
            fontsize=15, fontweight="bold", color=tc.PALETTE[key])
    ax.text(i + 0.45, 0.5, label, ha="center", va="center",
            fontsize=10, color="white", fontweight="bold", wrap=True)
ax.set_xlim(-0.1, 3); ax.set_ylim(-0.5, 1.1)
ax.set_axis_off(); ax.set_title("Lecture deck palette", fontsize=13)
fig.tight_layout(); fig.savefig(str(tc.FIG_DIR / "00_palette.png"), dpi=150)
plt.close(fig); print("saved", str(tc.FIG_DIR / "00_palette.png"))

saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/00_palette.png


## The fixed inverse problem $D$ that tutorials 2-5 attack

We commit to **one** physics for the rest of the lecture: **sparse-view CT** with
only 40 angles and a bit of measurement noise. Here CT's Radon operator plays the
role of the **photoacoustic wave operator** -- linear, ill-posed, limited-view.

The raw inversion is **Filtered Back-Projection** (FBP). With so few angles it is
streaky and noisy: this is exactly the gap the prior $R$ must close. Every later
tutorial reconstructs $x$ from this **same** $y$ by choosing a different $R$.

In [4]:
phys = tc.ct_physics(angles=40, sigma=0.02)   # the FIXED data-fidelity D (Radon + noise)
y = phys(x)                                    # noisy sinogram (measurements)  -- __call__ adds noise
x_fbp = phys.fbp(y)                            # fast filtered back-projection (no prior)
print("y (sinogram) shape:", tuple(y.shape))

tc.save_images(
    [x, y, x_fbp],
    titles=["x (object)", "y (sinogram)", tc.title_psnr("FBP(y), no prior", x_fbp, x)],
    fname="00_problem.png",
    figsize=(9, 4),     # wider so the two-line PSNR title does not overlap neighbours
)

y (sinogram) shape: (1, 1, 182, 40)
saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/00_problem.png


## The helper readout

One metric for the whole lecture: PSNR (higher = better). `tc.psnr(estimate, reference)`
turns "visibly better" into a single number we can put in every figure title.

In [5]:
print(f"FBP baseline PSNR: {tc.psnr(x_fbp, x):.2f} dB   <-- the bar every prior must beat")

FBP baseline PSNR: 15.23 dB   <-- the bar every prior must beat


## Takeaway

One object $x$, one operator family $D$ (sparse-view CT, our PAT stand-in), one metric (PSNR).
From here on the data-fidelity $D$ never changes -- **only the prior $R$ does.**